# Supervisor 멀티 에이전트 — 관리자가 작업을 배분

**Supervisor(관리자)** 구조는 Network 와 달리 **중앙 관리자가 작업을 배분** 한다. 작업자들은 서로 직접 대화하지 않고, 항상 supervisor 를 거친다.

```
          ┌──────────── supervisor ◀───────────┐
          │         (다음 작업자 선택)            │
          ▼                                      │
   search_agent / file_agent ──(작업 후 복귀)────┘
          (supervisor 가 FINISH 하면 END)
```

예제: 여행 정보(웹검색 + 날씨)를 조사하는 **search_agent**, 결과를 파일로 저장하는 **file_agent**, 이 둘을 지휘하는 **supervisor**.

핵심: supervisor 가 [basics] **구조화 출력(`Router`)** 으로 "다음에 누구를 시킬지"를 정한다.

> `OPENAI_API_KEY`, `TAVILY_API_KEY` 필요. 날씨는 `OPENWEATHERMAP_API_KEY` 가 있을 때만 활성화(없으면 자동 비활성).

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
for k in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(k), f"{k} 가 .env 에 없습니다"
# OPENWEATHERMAP_API_KEY 는 선택 — 있으면 날씨 도구 활성화
HAS_WEATHER = bool(os.environ.get("OPENWEATHERMAP_API_KEY"))
print("환경변수 로드 완료. 날씨 도구:", "활성" if HAS_WEATHER else "비활성(키 없음)")

## 1. search_agent — 웹검색 + (선택)날씨

[basics] `create_react_agent` 에 도구 여러 개를 준다. 날씨 도구는 OpenWeatherMap 키가 있을 때만 추가한다.

> 날씨 도구는 외부 유료 API(OpenWeatherMap One Call) 예시다. 키가 없으면 웹검색만으로 동작한다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
import requests
import json

tavily_tool = TavilySearchResults(max_results=5)

@tool
def weather_search(city: str) -> dict:
    """Get current + daily(8d) forecast and overview for a city.

    Args:
        city (str): the city name
    """
    api_key = os.getenv("OPENWEATHERMAP_API_KEY")
    # 좌표 조회
    geo = requests.get(
        f"http://api.openweathermap.org/geo/1.0/direct?q={city}&limit=1&appid={api_key}"
    ).json()
    if not geo:
        raise ValueError(f"도시 '{city}' 정보를 찾을 수 없습니다.")
    lat, lon = geo[0]["lat"], geo[0]["lon"]
    # 현재/일별 예보
    data = requests.get(
        f"https://api.openweathermap.org/data/3.0/onecall?lat={lat}&lon={lon}&lang=kr&appid={api_key}"
    ).json()
    overview = requests.get(
        f"https://api.openweathermap.org/data/3.0/onecall/overview?lat={lat}&lon={lon}&appid={api_key}"
    ).json()
    return {
        "current_weather": data.get("current"),
        "daily_weather": data.get("daily"),
        "weather_overview": overview.get("weather_overview"),
    }

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o")

# 날씨 키가 있을 때만 도구에 포함
search_tools = [tavily_tool] + ([weather_search] if HAS_WEATHER else [])
search_agent = create_react_agent(
    llm,
    tools=search_tools,
    prompt="You are a skilled information-seeking agent with web search"
           + (" and weather lookup" if HAS_WEATHER else "")
           + " tools to provide accurate, up-to-date information.",
)

## 2. file_agent — 결과를 파일로 저장
[basics] 단순 파일 저장 도구를 가진 에이전트.

In [ ]:
@tool
def save_file(content: str, output_path: str = "file_info.md") -> str:
    """Write the provided content to a text file at output_path."""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)
    return output_path

file_agent = create_react_agent(
    llm,
    [save_file],
    prompt="You are a file management agent for saving files upon request.",
)

## 3. Supervisor — 다음 작업자 선택

supervisor 는 대화를 보고 **다음에 일할 작업자** 를 고르거나 `FINISH` 한다. [basics] `with_structured_output(Router)` 로 선택지를 **정해진 값(작업자 이름 / FINISH)** 으로만 내게 강제한다.

In [ ]:
from langgraph.graph import MessagesState
from typing import Literal, TypedDict

members = ["search_agent", "file_agent"]
options = members + ["FINISH"]

class Router(TypedDict):
    """다음에 라우팅할 작업자. 필요 없으면 FINISH."""
    next: Literal[*options]

class State(MessagesState):
    next: str   # supervisor 가 정한 다음 행선지

In [ ]:
from langgraph.types import Command
from langgraph.graph import END

system_prompt = (
    "You are a supervisor managing a conversation between these workers: "
    f"{members}. Given the user request, respond with the worker to act next. "
    "Each worker performs a task and reports results. When finished, respond with FINISH."
)

def supervisor_node(state: State) -> Command[Literal[*members, "__end__"]]:
    messages = [{"role": "system", "content": system_prompt}] + state["messages"]
    response = llm.with_structured_output(Router).invoke(messages)
    goto = response["next"]
    if goto == "FINISH":
        goto = END
    return Command(goto=goto, update={"next": goto})

## 4. 작업자 노드 — 작업 후 supervisor 복귀

[basics] 각 작업자는 일을 마치면 결과를 메시지에 담고 **항상 `goto="supervisor"`** 로 돌아간다 (Network 와 달리 작업자끼리 직접 안 넘김).

In [ ]:
from langchain_core.messages import HumanMessage

def search_node(state: State) -> Command[Literal["supervisor"]]:
    result = search_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="search_agent")]},
        goto="supervisor",
    )

def file_node(state: State) -> Command[Literal["supervisor"]]:
    result = file_agent.invoke(state)
    return Command(
        update={"messages": [HumanMessage(content=result["messages"][-1].content, name="file_agent")]},
        goto="supervisor",
    )

## 5. 그래프 조립
[basics] START → supervisor 만 고정 엣지. 작업자↔supervisor 왕복은 `Command(goto)` 가 만든다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

graph_builder = StateGraph(State)
graph_builder.add_edge(START, "supervisor")
graph_builder.add_node("supervisor", supervisor_node)
graph_builder.add_node("search_agent", search_node)
graph_builder.add_node("file_agent", file_node)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트

여행 정보 조사 → 파일 저장 흐름. supervisor 가 search_agent → file_agent 순으로 배분하고 끝나면 FINISH 한다.

[basics] `stream_mode="updates"` + `subgraphs=True` 로 내부 에이전트 진행도 본다.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

user_input = (
    "제주도 여행을 계획 중이야. 가볼 만한 장소를 조사하고, "
    "그 내용을 jeju_travel.md 파일로 저장해줘."
)

for namespace, chunk in graph.stream(
    {"messages": user_input},
    stream_mode="updates",
    subgraphs=True,
    config=config,
):
    for node_name, node_chunk in chunk.items():
        if isinstance(node_chunk, dict) and "messages" in node_chunk:
            node_chunk["messages"][-1].pretty_print()
        else:
            print(node_name, node_chunk)

## 정리

- **Supervisor** = 중앙 관리자가 작업자에게 작업을 **배분**, 작업자는 항상 관리자로 복귀
- supervisor 가 `with_structured_output(Router)` 로 **다음 작업자 or FINISH** 를 결정
- 작업자 노드는 `Command(goto="supervisor")` 로 복귀 — 작업자끼리 직접 안 넘김 (Network 와의 차이)
- 한 작업자(search_agent)가 도구 여러 개(웹검색 + 날씨)를 가질 수 있다

| | Network | Supervisor |
|---|---|---|
| 흐름 | 에이전트끼리 직접 핸드오프 | 항상 관리자 경유 |
| 제어 | 분산 | 중앙집중 |
| 종료 | FINAL ANSWER | supervisor 의 FINISH |

다음: 팀(서브그래프)을 상위 관리자가 지휘하는 **Hierarchical** 구조.